# Ejercicio 7: Bases de Datos Vectoriales

### Nombre: Anthony Goyes

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [1]:
%pip install kagglehub pandas numpy tqdm sentence-transformers scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

In [3]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [4]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [5]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        
        if len(chunk) > 0:
            chunks.append(chunk)
        
        if end == n:
            break
        
        start = max(0, end - overlap)
    
    return chunks

In [6]:
MAX_DOCS = 300

df_sample = df.head(MAX_DOCS).copy()

records = []

for i, row in df_sample.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": int(j),
            "text": ch,
            "source": "wikipedia"
        })

chunks_df = pd.DataFrame(records)

print("Documentos usados:", len(df_sample))
print("Total de chunks:", len(chunks_df))

chunks_df.head()

Documentos usados: 300
Total de chunks: 1976


,doc_id,chunk_id,text,source
0,0,0,Anovo Anovo (formerly A Novo) is a computer se...,wikipedia
1,1,0,Battery indicator A battery indicator (also kn...,wikipedia
2,1,1,ad battery when in reality it indicates a prob...,wikipedia
3,1,2,s that an internal standby battery needs repla...,wikipedia
4,1,3,increase; in many cases the EMF remains more o...,wikipedia


In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-small-v2"

model = SentenceTransformer(MODEL_NAME)

print("Modelo cargado correctamente:", MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado correctamente: intfloat/e5-small-v2


In [8]:
# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

print("Total de passages:", len(passages))
print(passages[0][:500])

Total de passages: 1976
passage: Anovo Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It was founded in 1987, went public in 1999, and is currently a member of the CAC Small. It won in the category 'Service and Repair' of the Mobile News Awards four years in a row, from 2007 to 2010. As of November 2017, they have a score of 1.6 out of 10 on the TrustPilot ratings site, with 86% of reviewers giving the company the lowest possible rating.


In [9]:
# Embeddings (N x D)
# Se usa normalize_embeddings=True para similitud coseno

embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embeddings generados correctamente")
print("Shape:", embeddings.shape)
print("Tipo:", embeddings.dtype)

Batches:   0%|          | 0/124 [00:00<?, ?it/s]

Embeddings generados correctamente
Shape: (1976, 384)
Tipo: float32


In [10]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    
    return vec

query_text = "Battery measuring"

query_embedding = embed_query(query_text)

print(query_embedding.shape)

(1, 384)


Para reducir el tiempo de descarga y ejecución, se utilizó el modelo `intfloat/e5-small-v2`. Este modelo pertenece a la familia E5 y sirve para tareas de recuperación semántica usando los prefijos `query:` para consultas y `passage:` para documentos. En este caso, los embeddings generados tienen dimensión 384.

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [11]:
%pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importar FAISS

In [12]:
import faiss
import numpy as np

print("FAISS importado correctamente")

FAISS importado correctamente


### Crear índice FAISS

In [13]:
D = embeddings.shape[1]

index_faiss = faiss.IndexFlatIP(D)

index_faiss.add(embeddings)

print("Dimensión de los embeddings:", D)
print("Vectores indexados:", index_faiss.ntotal)


Dimensión de los embeddings: 384
Vectores indexados: 1976


### Busqueda Simple

In [14]:
k = 5

scores, indices = index_faiss.search(query_embedding, k)

print("Scores:")
print(scores)

print("Índices:")
print(indices)

Scores:
[[0.8954438  0.8741763  0.8488823  0.84366417 0.84231865]]
Índices:
[[   1    2    4 1753    5]]


### Ver los textos recuperados

In [15]:
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    row = chunks_df.iloc[idx]
    
    print("=" * 100)
    print("Rank:", rank)
    print("Índice FAISS:", int(idx))
    print("Score:", float(score))
    print("Doc ID:", row["doc_id"])
    print("Chunk ID:", row["chunk_id"])
    print("Source:", row["source"])
    print()
    print(row["text"][:700])

Rank: 1
Índice FAISS: 1
Score: 0.8954437971115112
Doc ID: 1
Chunk ID: 0
Source: wikipedia

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
Rank: 2
Índice FAISS: 2
Score: 0.8741763234138489
Doc ID: 1
Chunk ID: 1
Source: wikipedia

ad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively, an ammeter may b

### Función reutilizable

In [16]:
def faiss_search(query_embedding, k=5):
    scores, indices = index_faiss.search(query_embedding, k)
    
    results = []
    
    for score, idx in zip(scores[0], indices[0]):
        row = chunks_df.iloc[idx]
        
        results.append({
            "id": int(idx),
            "score": float(score),
            "text": row["text"],
            "metadata": {
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"]),
                "source": row["source"]
            }
        })
    
    return results

In [17]:
faiss_results = faiss_search(query_embedding, k=5)

for r in faiss_results:
    print("=" * 100)
    print("ID:", r["id"])
    print("Score:", r["score"])
    print("Metadata:", r["metadata"])
    print()
    print(r["text"][:700])

ID: 1
Score: 0.8954437971115112
Metadata: {'doc_id': 1, 'chunk_id': 0, 'source': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 2
Score: 0.8741763234138489
Metadata: {'doc_id': 1, 'chunk_id': 1, 'source': 'wikipedia'}

ad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively, an ammet

### Mostrar en tabla

In [18]:
faiss_df = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "doc_id": r["metadata"]["doc_id"],
        "chunk_id": r["metadata"]["chunk_id"],
        "source": r["metadata"]["source"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(faiss_results)
])

faiss_df

,rank,id,score,doc_id,chunk_id,source,text
0,1,1,0.895444,1,0,wikipedia,Battery indicator A battery indicator (also kn...
1,2,2,0.874176,1,1,wikipedia,ad battery when in reality it indicates a prob...
2,3,4,0.848882,1,3,wikipedia,increase; in many cases the EMF remains more o...
3,4,1753,0.843664,257,9,wikipedia,mping around large conductors and bus bars up ...
4,5,5,0.842319,1,4,wikipedia,"otective diodes cannot be used, a battery will..."


En esta sección se utilizó FAISS para construir un índice vectorial en memoria a partir de los embeddings generados previamente. Cada embedding representa un chunk del corpus de Wikipedia.

Se utilizó `IndexFlatIP`, que realiza búsqueda por producto interno. Como los embeddings fueron generados con `normalize_embeddings=True`, el producto interno se puede interpretar como similitud coseno.

FAISS devuelve los índices de los vectores más similares a la consulta. Luego, esos índices se usan para recuperar el texto y la metadata correspondiente desde `chunks_df`.

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [19]:
%pip install qdrant-client

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importar Qdrant

In [20]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Qdrant importado correctamente")

Qdrant importado correctamente


### Cliente local en memeoria

In [21]:
qdrant_client = QdrantClient(":memory:")

print("Cliente Qdrant creado en memoria")

Cliente Qdrant creado en memoria


### Crear coleccion

In [22]:
collection_name = "wiki_chunks"

vector_size = embeddings.shape[1]

if qdrant_client.collection_exists(collection_name):
    qdrant_client.delete_collection(collection_name)

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE
    )
)

print("Colección creada:", collection_name)
print("Dimensión vectorial:", vector_size)

Colección creada: wiki_chunks
Dimensión vectorial: 384


### Crear puntos para para insertar

In [23]:
points = []

for idx, row in chunks_df.iterrows():
    point = PointStruct(
        id=int(idx),
        vector=embeddings[idx].tolist(),
        payload={
            "text": row["text"],
            "doc_id": int(row["doc_id"]),
            "chunk_id": int(row["chunk_id"]),
            "source": row["source"]
        }
    )
    
    points.append(point)

print("Total de puntos creados:", len(points))

Total de puntos creados: 1976


### Insertar puntos en Qdrant

In [24]:
qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

print("Puntos insertados en Qdrant:", len(points))

Puntos insertados en Qdrant: 1976


### Verificar coleccion

In [25]:
collection_info = qdrant_client.get_collection(collection_name)

print(collection_info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=1976 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimized=None), wal_config=W

### Buscar con Qdrant

In [26]:
k = 5

qdrant_response = qdrant_client.query_points(
    collection_name=collection_name,
    query=query_embedding[0].tolist(),
    limit=k
)

qdrant_response.points

[ScoredPoint(id=1, version=0, score=0.8954437503867796, payload={'text': "Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a bad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively,", 'doc_id': 1, 'chunk_id': 0, 'source': 'wikipedia'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, vers

### Ver resultados de forma visible

In [27]:
for rank, point in enumerate(qdrant_response.points, start=1):
    print("=" * 100)
    print("Rank:", rank)
    print("ID:", point.id)
    print("Score:", point.score)
    print("Doc ID:", point.payload["doc_id"])
    print("Chunk ID:", point.payload["chunk_id"])
    print("Source:", point.payload["source"])
    print()
    print(point.payload["text"][:700])

Rank: 1
ID: 1
Score: 0.8954437503867796
Doc ID: 1
Chunk ID: 0
Source: wikipedia

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
Rank: 2
ID: 2
Score: 0.8741762636528344
Doc ID: 1
Chunk ID: 1
Source: wikipedia

ad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively, an ammeter may be fitted. This indic

### Funcion reutilizable

In [28]:
def qdrant_search(query_embedding, k=5):
    response = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding[0].tolist(),
        limit=k
    )
    
    output = []
    
    for point in response.points:
        output.append({
            "id": point.id,
            "score": float(point.score),
            "text": point.payload["text"],
            "metadata": {
                "doc_id": point.payload["doc_id"],
                "chunk_id": point.payload["chunk_id"],
                "source": point.payload["source"]
            }
        })
    
    return output

In [29]:
qdrant_results = qdrant_search(query_embedding, k=5)

for r in qdrant_results:
    print("=" * 100)
    print("ID:", r["id"])
    print("Score:", r["score"])
    print("Metadata:", r["metadata"])
    print()
    print(r["text"][:700])

ID: 1
Score: 0.8954437503867796
Metadata: {'doc_id': 1, 'chunk_id': 0, 'source': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 2
Score: 0.8741762636528344
Metadata: {'doc_id': 1, 'chunk_id': 1, 'source': 'wikipedia'}

ad battery when in reality it indicates a problem with the vehicle's charging system. Alternatively, an ammet

### Resultados en tabla

In [30]:
qdrant_df = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "doc_id": r["metadata"]["doc_id"],
        "chunk_id": r["metadata"]["chunk_id"],
        "source": r["metadata"]["source"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(qdrant_results)
])

qdrant_df

,rank,id,score,doc_id,chunk_id,source,text
0,1,1,0.895444,1,0,wikipedia,Battery indicator A battery indicator (also kn...
1,2,2,0.874176,1,1,wikipedia,ad battery when in reality it indicates a prob...
2,3,4,0.848882,1,3,wikipedia,increase; in many cases the EMF remains more o...
3,4,1753,0.843664,257,9,wikipedia,mping around large conductors and bus bars up ...
4,5,5,0.842319,1,4,wikipedia,"otective diodes cannot be used, a battery will..."


### FAISS vs Qdrant

In [31]:
comparison_df = pd.DataFrame({
    "rank": range(1, 6),
    "faiss_id": [r["id"] for r in faiss_results],
    "faiss_score": [r["score"] for r in faiss_results],
    "qdrant_id": [r["id"] for r in qdrant_results],
    "qdrant_score": [r["score"] for r in qdrant_results],
})

comparison_df

,rank,faiss_id,faiss_score,qdrant_id,qdrant_score
0,1,1,0.895444,1,0.895444
1,2,2,0.874176,2,0.874176
2,3,4,0.848882,4,0.848882
3,4,1753,0.843664,1753,0.843664
4,5,5,0.842319,5,0.842319


### Prueba con otra consulta

In [32]:
query_text_2 = "computer repair company"

query_embedding_2 = embed_query(query_text_2)

qdrant_results_2 = qdrant_search(query_embedding_2, k=5)

qdrant_df_2 = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "doc_id": r["metadata"]["doc_id"],
        "chunk_id": r["metadata"]["chunk_id"],
        "source": r["metadata"]["source"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(qdrant_results_2)
])

qdrant_df_2

,rank,id,score,doc_id,chunk_id,source,text
0,1,0,0.832882,0,0,wikipedia,Anovo Anovo (formerly A Novo) is a computer se...
1,2,1118,0.830225,169,0,wikipedia,Fujitsu Computer Products of America Fujitsu C...
2,3,1771,0.814612,260,0,wikipedia,Data Processing Iran Co. Data Processing Iran ...
3,4,874,0.810400,148,0,wikipedia,ASBIS ASBISC Enterprises PLC is a multinationa...
4,5,1119,0.809864,169,1,wikipedia,engineering and technical support for the Fuji...


En esta sección se utilizó Qdrant como base de datos vectorial. A diferencia de FAISS, Qdrant permite almacenar cada vector junto con metadata o payload. Esto facilita recuperar no solo el ID del vector, sino también información adicional como el texto original, el `doc_id`, el `chunk_id` y la fuente.

Se creó una colección llamada `wiki_chunks`, configurada con distancia coseno. Esta métrica fue elegida porque los embeddings se generaron con `normalize_embeddings=True`, por lo que resulta adecuada para comparar similitud semántica.

Luego se insertaron los embeddings como puntos de Qdrant. Cada punto contiene un vector y un payload con el texto y sus metadatos. Finalmente, se realizó una consulta vectorial usando `query_points`, recuperando los chunks más similares a la consulta.

### Preguntas Qdrant

**¿La métrica usada fue cosine o L2? ¿Por qué?**  
Se utilizó la métrica cosine porque los embeddings fueron normalizados usando `normalize_embeddings=True`. Esto permite comparar la similitud semántica entre documentos y consultas mediante la cercanía angular de sus vectores.

**¿Qué tan fácil fue filtrar o trabajar con metadata en comparación con FAISS?**  
En Qdrant es más sencillo trabajar con metadata porque cada vector se almacena junto con un payload. En FAISS, en cambio, el índice solo almacena vectores, por lo que la metadata debe mantenerse de forma externa en un DataFrame como `chunks_df`.

**¿Qué pasa con el tiempo de respuesta cuando aumentas k?**  
Al aumentar `k`, se recuperan más resultados, por lo que el tiempo de respuesta puede aumentar ligeramente. Sin embargo, con una muestra pequeña como la usada en esta práctica, la diferencia suele ser baja.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


### Instalar Milvus


In [33]:
%pip install -U "pymilvus[milvus-lite]"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
%pip install -U "pymilvus[milvus_lite]"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
%pip install -U milvus-lite

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importar y conectar a Milvus


In [41]:
from pathlib import Path
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

MILVUS_DB_PATH = Path(f"./milvus_wiki_{run_id}.db")

print("Base local de Milvus para esta ejecución:", MILVUS_DB_PATH)

Base local de Milvus para esta ejecución: milvus_wiki_20260706_083946.db


In [42]:
from pymilvus import MilvusClient, DataType
import time
import pandas as pd

milvus_client = MilvusClient("./milvus_wiki.db")

print("Conectado a Milvus Lite")

Conectado a Milvus Lite


### Preparar datos

In [43]:
D = embeddings.shape[1]
N = len(chunks_df)

milvus_data = []

for idx, row in chunks_df.iterrows():
    milvus_data.append({
        "id": int(idx),
        "embedding": embeddings[idx].tolist(),
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"]),
        "source": str(row["source"])[:500],
        "title": str(row["source"])[:500],
        "category": "wikipedia",
        "text": str(row["text"])[:3000]
    })

print("Dimensión:", D)
print("Registros preparados:", len(milvus_data))

Dimensión: 384
Registros preparados: 1976


### Función para colección Milvus

In [44]:
def create_milvus_collection(collection_name, index_type="FLAT"):
    if milvus_client.has_collection(collection_name):
        milvus_client.drop_collection(collection_name)
    
    schema = MilvusClient.create_schema(
        auto_id=False,
        enable_dynamic_field=False
    )
    
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=D)
    schema.add_field(field_name="doc_id", datatype=DataType.INT64)
    schema.add_field(field_name="chunk_id", datatype=DataType.INT64)
    schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=512)
    schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=512)
    schema.add_field(field_name="category", datatype=DataType.VARCHAR, max_length=100)
    schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=4000)
    
    index_params = milvus_client.prepare_index_params()
    
    if index_type == "FLAT":
        index_params.add_index(
            field_name="embedding",
            index_type="FLAT",
            metric_type="COSINE"
        )
    elif index_type == "HNSW":
        index_params.add_index(
            field_name="embedding",
            index_type="HNSW",
            metric_type="COSINE",
            params={
                "M": 16,
                "efConstruction": 100
            }
        )
    
    milvus_client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )
    
    print("Colección creada:", collection_name, "| Índice:", index_type)

### Crear colecciones

In [45]:
collection_flat = "wiki_milvus_flat"
collection_hnsw = "wiki_milvus_hnsw"

create_milvus_collection(collection_flat, index_type="FLAT")
create_milvus_collection(collection_hnsw, index_type="HNSW")

Colección creada: wiki_milvus_flat | Índice: FLAT
Colección creada: wiki_milvus_hnsw | Índice: HNSW


### Insertar embeddings

In [46]:
milvus_client.insert(
    collection_name=collection_flat,
    data=milvus_data
)

milvus_client.insert(
    collection_name=collection_hnsw,
    data=milvus_data
)

print("Datos insertados en ambas colecciones:", len(milvus_data))

Datos insertados en ambas colecciones: 1976


### Cargar colecciones

In [47]:
milvus_client.load_collection(collection_flat)
milvus_client.load_collection(collection_hnsw)

print("Colecciones cargadas en memoria")

Colecciones cargadas en memoria


### Función de búsqueda Milvus

In [48]:
def milvus_search(query_embedding, k=5, collection_name=collection_hnsw, search_params=None):
    if search_params is None:
        search_params = {
            "metric_type": "COSINE",
            "params": {
                "ef": 32
            }
        }
    
    response = milvus_client.search(
        collection_name=collection_name,
        data=query_embedding.tolist(),
        anns_field="embedding",
        search_params=search_params,
        limit=k,
        output_fields=["doc_id", "chunk_id", "source", "title", "category", "text"]
    )
    
    results = []
    
    for hit in response[0]:
        entity = hit["entity"]
        
        results.append({
            "id": hit["id"],
            "score": hit["distance"],
            "text": entity["text"],
            "metadata": {
                "doc_id": entity["doc_id"],
                "chunk_id": entity["chunk_id"],
                "source": entity["source"],
                "title": entity["title"],
                "category": entity["category"]
            }
        })
    
    return results

### Probar búsqueda Top-k


In [49]:
milvus_results = milvus_search(query_embedding, k=5)

for r in milvus_results:
    print("=" * 100)
    print("ID:", r["id"])
    print("Score:", r["score"])
    print("Metadata:", r["metadata"])
    print()
    print(r["text"][:700])

ID: 1
Score: 0.10455632209777832
Metadata: {'doc_id': 1, 'chunk_id': 0, 'source': 'wikipedia', 'title': 'wikipedia', 'category': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 2
Score: 0.12582379579544067
Metadata: {'doc_id': 1, 'chunk_id': 1, 'source': 'wikipedia', 'title': 'wikipedia', 'category': 'wikipedia'}

ad battery wh

### Búsqueda exacta vs ANN

In [50]:
def timed_milvus_search(collection_name, query_embedding, k, search_params):
    start = time.perf_counter()
    
    results = milvus_search(
        query_embedding=query_embedding,
        k=k,
        collection_name=collection_name,
        search_params=search_params
    )
    
    elapsed = time.perf_counter() - start
    
    return results, elapsed

In [51]:
flat_params = {
    "metric_type": "COSINE",
    "params": {}
}

hnsw_fast_params = {
    "metric_type": "COSINE",
    "params": {
        "ef": 10
    }
}

hnsw_precise_params = {
    "metric_type": "COSINE",
    "params": {
        "ef": 64
    }
}

In [52]:
experiment_rows = []

for k in [5, 20]:
    flat_results, flat_time = timed_milvus_search(
        collection_flat,
        query_embedding,
        k,
        flat_params
    )
    
    hnsw_fast_results, hnsw_fast_time = timed_milvus_search(
        collection_hnsw,
        query_embedding,
        k,
        hnsw_fast_params
    )
    
    hnsw_precise_results, hnsw_precise_time = timed_milvus_search(
        collection_hnsw,
        query_embedding,
        k,
        hnsw_precise_params
    )
    
    flat_ids = set(r["id"] for r in flat_results)
    hnsw_fast_ids = set(r["id"] for r in hnsw_fast_results)
    hnsw_precise_ids = set(r["id"] for r in hnsw_precise_results)
    
    experiment_rows.append({
        "k": k,
        "flat_time_seconds": flat_time,
        "hnsw_fast_time_seconds": hnsw_fast_time,
        "hnsw_precise_time_seconds": hnsw_precise_time,
        "overlap_flat_vs_hnsw_fast": len(flat_ids.intersection(hnsw_fast_ids)),
        "overlap_flat_vs_hnsw_precise": len(flat_ids.intersection(hnsw_precise_ids)),
        "flat_ids": list(flat_ids),
        "hnsw_fast_ids": list(hnsw_fast_ids),
        "hnsw_precise_ids": list(hnsw_precise_ids)
    })

milvus_experiment_df = pd.DataFrame(experiment_rows)
milvus_experiment_df

,k,flat_time_seconds,hnsw_fast_time_seconds,hnsw_precise_time_seconds,overlap_flat_vs_hnsw_fast,overlap_flat_vs_hnsw_precise,flat_ids,hnsw_fast_ids,hnsw_precise_ids
0,5,0.360476,0.360010,0.416589,5,5,"[1, 2, 4, 5, 1753]","[1, 2, 4, 5, 1753]","[1, 2, 4, 5, 1753]"
1,20,0.335929,0.421192,0.383177,20,20,"[1, 2, 3, 4, 5, 774, 14, 1550, 1433, 1434, 174...","[1, 2, 3, 4, 5, 774, 14, 1550, 1433, 1434, 174...","[1, 2, 3, 4, 5, 774, 14, 1550, 1433, 1434, 174..."


### Tabla de resultados Milvus

In [53]:
milvus_df = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "doc_id": r["metadata"]["doc_id"],
        "chunk_id": r["metadata"]["chunk_id"],
        "source": r["metadata"]["source"],
        "category": r["metadata"]["category"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(milvus_results)
])

milvus_df

,rank,id,score,doc_id,chunk_id,source,category,text
0,1,1,0.104556,1,0,wikipedia,wikipedia,Battery indicator A battery indicator (also kn...
1,2,2,0.125824,1,1,wikipedia,wikipedia,ad battery when in reality it indicates a prob...
2,3,4,0.151118,1,3,wikipedia,wikipedia,increase; in many cases the EMF remains more o...
3,4,1753,0.156336,257,9,wikipedia,wikipedia,mping around large conductors and bus bars up ...
4,5,5,0.157681,1,4,wikipedia,wikipedia,"otective diodes cannot be used, a battery will..."


En esta sección se implementó Milvus como segunda base de datos vectorial. Se utilizó Milvus Lite para trabajar localmente desde VS Code sin necesidad de levantar un servidor externo.

Se crearon dos colecciones con el mismo esquema: una colección con índice `FLAT`, usada como búsqueda exacta o más precisa, y otra colección con índice `HNSW`, usada como búsqueda aproximada ANN. Cada registro contiene un identificador, el embedding, texto y metadata como `doc_id`, `chunk_id`, `source`, `title` y `category`.

Para comparar precisión y velocidad, se ejecutaron consultas con `k=5` y `k=20`. En la configuración HNSW se ajustó el parámetro `ef`: valores bajos favorecen velocidad, mientras que valores más altos suelen mejorar la precisión al explorar más candidatos durante la búsqueda.

### Preguntas Milvus

**¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**  
Se utilizó el índice `HNSW` y se ajustó el parámetro `ef` durante la búsqueda. Con `ef=10` se priorizó velocidad, mientras que con `ef=64` se buscó mayor precisión. Además, se comparó contra una colección con índice `FLAT`, que sirve como referencia más exacta.

**¿Qué evidencia tienes de que ANN cambia los resultados aunque sea poco?**  
La evidencia se obtiene comparando los IDs recuperados por `FLAT` contra los IDs recuperados por `HNSW`. Para esto se calculó el overlap de resultados en `milvus_experiment_df`. Si el overlap es menor que `k`, significa que la búsqueda ANN devolvió algunos resultados diferentes. Si el overlap es igual a `k`, entonces para esta muestra pequeña ANN recuperó los mismos resultados que la búsqueda exacta.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [ ]:
%pip install -U weaviate-client

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/652.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/652.7 kB ? eta -:--:--
   ---------------------------------------- 652.7/652.7 kB 3.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.8 MB ? eta -:--:--
   ------------- -------------------------- 1.6/4.8 MB 8.2 MB/s eta 0:00:01
   ------------------------------ --------- 3.7/4.8 MB 8.7 MB/s eta 0:00:01
   ---------------------------------------- 4.8/4.8 MB 8.1 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.80.0
    Uninstalling grpcio-1.80.0:
      Successfully uninstalled grpcio-1.80.0
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
import weaviate
import weaviate.classes as wvc
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery, Filter
import pandas as pd
import time

weaviate_client = weaviate.connect_to_local()

print("Weaviate listo:", weaviate_client.is_ready())

Weaviate listo: True


### Crear colección

In [56]:
collection_name = "Document"

if weaviate_client.collections.exists(collection_name):
    weaviate_client.collections.delete(collection_name)

weaviate_client.collections.create(
    name=collection_name,
    vector_config=Configure.Vectors.self_provided(),
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="title", data_type=DataType.TEXT),
        Property(name="category", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="docId", data_type=DataType.INT),
        Property(name="chunkId", data_type=DataType.INT),
        Property(name="chunkIndex", data_type=DataType.INT),
    ]
)

documents_collection = weaviate_client.collections.use(collection_name)

print("Colección creada:", collection_name)

Colección creada: Document


### Preparar objetos

In [57]:
weaviate_objects = []

for idx, row in chunks_df.iterrows():
    weaviate_objects.append(
        wvc.data.DataObject(
            properties={
                "text": str(row["text"]),
                "title": str(row["source"]),
                "category": "wikipedia",
                "source": str(row["source"]),
                "docId": int(row["doc_id"]),
                "chunkId": int(row["chunk_id"]),
                "chunkIndex": int(idx),
            },
            vector=embeddings[idx].tolist()
        )
    )

print("Objetos preparados:", len(weaviate_objects))

Objetos preparados: 1976


### Insertar objetos

In [58]:
insert_response = documents_collection.data.insert_many(weaviate_objects)

print("Objetos insertados:", len(weaviate_objects))
print("Errores:", insert_response.has_errors)

Objetos insertados: 1976
Errores: False


### Búsqueda simple

In [59]:
response = documents_collection.query.near_vector(
    near_vector=query_embedding[0].tolist(),
    limit=5,
    return_metadata=MetadataQuery(distance=True)
)

for i, obj in enumerate(response.objects, start=1):
    print("=" * 100)
    print("Rank:", i)
    print("UUID:", obj.uuid)
    print("Distance:", obj.metadata.distance)
    print("Metadata:", {
        "docId": obj.properties["docId"],
        "chunkId": obj.properties["chunkId"],
        "source": obj.properties["source"],
        "category": obj.properties["category"]
    })
    print()
    print(obj.properties["text"][:700])

Rank: 1
UUID: d4676a96-ae60-4315-b61f-21fb890c3589
Distance: 0.10455620288848877
Metadata: {'docId': 1, 'chunkId': 0, 'source': 'wikipedia', 'category': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
Rank: 2
UUID: 823c76bb-ebfd-4245-a5cb-f91a3802537f
Distance: 0.12582355737686157
Metadata: {'docId': 1, 'chunkId': 1, 'source': 'wik

### Función entregable

In [61]:
def weaviate_search(query_embedding, k=5, category_filter=None):
    filters = None
    
    if category_filter is not None:
        filters = Filter.by_property("category").equal(category_filter)
    
    response = documents_collection.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        filters=filters,
        return_metadata=MetadataQuery(distance=True)
    )
    
    results = []
    
    for obj in response.objects:
        distance = float(obj.metadata.distance)
        
        results.append({
            "id": str(obj.uuid),
            "score": 1 - distance,
            "distance": distance,
            "text": obj.properties["text"],
            "metadata": {
                "chunkIndex": obj.properties["chunkIndex"],
                "docId": obj.properties["docId"],
                "chunkId": obj.properties["chunkId"],
                "title": obj.properties["title"],
                "source": obj.properties["source"],
                "category": obj.properties["category"]
            }
        })
    
    return results

### Probar función

In [62]:
weaviate_results = weaviate_search(query_embedding, k=5)

for r in weaviate_results:
    print("=" * 100)
    print("ID:", r["id"])
    print("Score:", r["score"])
    print("Distance:", r["distance"])
    print("Metadata:", r["metadata"])
    print()
    print(r["text"][:700])

ID: d4676a96-ae60-4315-b61f-21fb890c3589
Score: 0.8954437971115112
Distance: 0.10455620288848877
Metadata: {'chunkIndex': 1, 'docId': 1, 'chunkId': 0, 'title': 'wikipedia', 'source': 'wikipedia', 'category': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 823c76bb-ebfd-4245-a5cb-f91a3802537f
Score: 0.8741764426231384
Distance: 

### Tabla

In [63]:
weaviate_df = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "distance": r["distance"],
        "docId": r["metadata"]["docId"],
        "chunkId": r["metadata"]["chunkId"],
        "source": r["metadata"]["source"],
        "category": r["metadata"]["category"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(weaviate_results)
])

weaviate_df

,rank,id,score,distance,docId,chunkId,source,category,text
0,1,d4676a96-ae60-4315-b61f-21fb890c3589,0.895444,0.104556,1,0,wikipedia,wikipedia,Battery indicator A battery indicator (also kn...
1,2,823c76bb-ebfd-4245-a5cb-f91a3802537f,0.874176,0.125824,1,1,wikipedia,wikipedia,ad battery when in reality it indicates a prob...
2,3,1513f559-2632-458f-a03a-9865e68d8500,0.848882,0.151118,1,3,wikipedia,wikipedia,increase; in many cases the EMF remains more o...
3,4,02b7c3ba-7c6c-4c8e-9ad0-855e846ef0b5,0.843664,0.156336,257,9,wikipedia,wikipedia,mping around large conductors and bus bars up ...
4,5,8abb50d7-2ca8-4cf6-86ab-057922489dba,0.842319,0.157681,1,4,wikipedia,wikipedia,"otective diodes cannot be used, a battery will..."


### Filtro opcional

In [64]:
weaviate_filtered_results = weaviate_search(
    query_embedding,
    k=5,
    category_filter="wikipedia"
)

len(weaviate_filtered_results)

5

In [65]:
weaviate_client.close()

## Parte 5: Weaviate

En esta sección se utilizó Weaviate como tercera base de datos vectorial. Se creó una colección llamada `Document` con propiedades como `text`, `title`, `category`, `source`, `docId` y `chunkId`.

Como los embeddings ya fueron generados previamente con el modelo E5, la colección se configuró con `self_provided`. Esto significa que Weaviate no genera los vectores automáticamente, sino que recibe los embeddings calculados externamente.

Luego se insertaron objetos compuestos por propiedades y vector. Finalmente, se ejecutó una búsqueda semántica Top-k usando `near_vector`, recuperando los documentos más cercanos al embedding de la consulta. También se probó un filtro por la propiedad `category`.

### Preguntas Weaviate

**¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**  
En una base relacional se trabaja con tablas y filas, donde cada fila sigue una estructura tabular. En Weaviate se trabaja con colecciones y objetos: cada objeto tiene propiedades descriptivas y además un vector asociado. Esto permite combinar búsqueda semántica con metadata dentro del mismo modelo de datos.

**¿Cómo describirías el trade-off de complejidad vs expresividad?**  
Weaviate puede ser más complejo que una tabla tradicional porque requiere definir colecciones, propiedades, vectorización y consultas semánticas. Sin embargo, ofrece mayor expresividad porque permite buscar por similitud vectorial, filtrar por metadata y representar documentos como objetos enriquecidos, no solo como filas.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [67]:
%pip install chromadb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\Anthony\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=6>
  res = process_handler(cmd, _system_body)
C:\Users\Anthony\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=7>
  res = process_handler(cmd, _system_body)
C:\Users\Anthony\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=8>
  res = process_handler(cmd, _system_body)


### Importar Chroma

In [68]:
import chromadb
import pandas as pd

print("Chroma importado correctamente")

Chroma importado correctamente


### Crear cliente y colección

In [69]:
chroma_client = chromadb.Client()

chroma_collection_name = "wiki_chunks_chroma"

try:
    chroma_client.delete_collection(chroma_collection_name)
except:
    pass

chroma_collection = chroma_client.create_collection(
    name=chroma_collection_name,
    metadata={"hnsw:space": "cosine"}
)

print("Colección creada en Chroma:", chroma_collection_name)


Colección creada en Chroma: wiki_chunks_chroma


### Preparar datos

In [70]:
ids = [str(i) for i in range(len(chunks_df))]

documents = chunks_df["text"].astype(str).tolist()

metadatas = [
    {
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"]),
        "source": str(row["source"]),
        "category": "wikipedia"
    }
    for _, row in chunks_df.iterrows()
]

print("IDs:", len(ids))
print("Documents:", len(documents))
print("Metadatas:", len(metadatas))
print("Embeddings:", embeddings.shape)

IDs: 1976
Documents: 1976
Metadatas: 1976
Embeddings: (1976, 384)


### Insertar datos

In [71]:
chroma_collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=documents,
    metadatas=metadatas
)

print("Datos insertados en Chroma:", len(ids))

Datos insertados en Chroma: 1976


### Consulta Top-k con k=5

In [72]:
k = 5

chroma_response = chroma_collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=k,
    include=["documents", "metadatas", "distances"]
)

for i in range(k):
    print("=" * 100)
    print("Rank:", i + 1)
    print("ID:", chroma_response["ids"][0][i])
    print("Distance:", chroma_response["distances"][0][i])
    print("Metadata:", chroma_response["metadatas"][0][i])
    print()
    print(chroma_response["documents"][0][i][:700])

Rank: 1
ID: 1
Distance: 0.10455608367919922
Metadata: {'source': 'wikipedia', 'doc_id': 1, 'category': 'wikipedia', 'chunk_id': 0}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
Rank: 2
ID: 2
Distance: 0.1258234977722168
Metadata: {'chunk_id': 1, 'doc_id': 1, 'category': 'wikipedia', 'source': 'wikipedia'}

ad battery when in reality it indica

### Funcion entregable

In [74]:
def chroma_search(query_embedding, k=5):
    response = chroma_collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    results = []
    
    for i in range(k):
        distance = float(response["distances"][0][i])
        
        results.append({
            "id": response["ids"][0][i],
            "score": 1 - distance,
            "distance": distance,
            "text": response["documents"][0][i],
            "metadata": response["metadatas"][0][i]
        })
    
    return results

### Probar función

In [75]:
chroma_results = chroma_search(query_embedding, k=5)

for r in chroma_results:
    print("=" * 100)
    print("ID:", r["id"])
    print("Score:", r["score"])
    print("Distance:", r["distance"])
    print("Metadata:", r["metadata"])
    print()
    print(r["text"][:700])

ID: 1
Score: 0.8954439163208008
Distance: 0.10455608367919922
Metadata: {'chunk_id': 0, 'category': 'wikipedia', 'doc_id': 1, 'source': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 2
Score: 0.8741765022277832
Distance: 0.1258234977722168
Metadata: {'source': 'wikipedia', 'chunk_id': 1, 'doc_id': 1, 'category': 'wikipedia'}



### Tabla

In [76]:
chroma_df = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "distance": r["distance"],
        "doc_id": r["metadata"]["doc_id"],
        "chunk_id": r["metadata"]["chunk_id"],
        "source": r["metadata"]["source"],
        "category": r["metadata"]["category"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(chroma_results)
])

chroma_df

,rank,id,score,distance,doc_id,chunk_id,source,category,text
0,1,1,0.895444,0.104556,1,0,wikipedia,wikipedia,Battery indicator A battery indicator (also kn...
1,2,2,0.874177,0.125823,1,1,wikipedia,wikipedia,ad battery when in reality it indicates a prob...
2,3,4,0.848882,0.151118,1,3,wikipedia,wikipedia,increase; in many cases the EMF remains more o...
3,4,1753,0.843664,0.156336,257,9,wikipedia,wikipedia,mping around large conductors and bus bars up ...
4,5,5,0.842319,0.157681,1,4,wikipedia,wikipedia,"otective diodes cannot be used, a battery will..."


## Parte 6: Chroma

En esta sección se utilizó Chroma como vector store ligero para prototipado rápido. Se creó una colección llamada `wiki_chunks_chroma`, configurada con distancia coseno, y se insertaron los chunks junto con sus embeddings, documentos originales y metadata.

A diferencia de Milvus y Weaviate, Chroma permitió reproducir el pipeline de búsqueda semántica sin levantar infraestructura adicional. La consulta se realizó directamente con `query_embeddings`, usando el embedding previamente generado para la consulta.

### Preguntas Chroma

**¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**  
Chroma fue más sencillo de implementar porque no necesitó Docker, servidor externo ni configuración avanzada de índices. En comparación con Milvus, requiere menos pasos de infraestructura. En comparación con Qdrant, también resulta simple porque se puede crear una colección e insertar documentos, embeddings y metadata directamente desde el notebook.

**¿Qué limitaciones ves para un sistema en producción?**  
Para producción, Chroma puede quedarse corto si se necesita alta escalabilidad, múltiples usuarios concurrentes, administración avanzada, monitoreo, despliegue distribuido o control fino sobre índices y rendimiento. Es muy útil para prototipos, pruebas locales y proyectos pequeños, pero para sistemas grandes conviene evaluar opciones como Milvus, Weaviate o Qdrant.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


### Instalar postgresql

In [78]:
%pip install "psycopg[binary]"

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.6 MB 4.7 MB/s eta 0:00:01
   ----------------------------- ---------- 2.6/3.6 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 3.6/3.6 MB 7.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\Anthony\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedWriter name=6>
  res = process_handler(cmd, _system_body)
C:\Users\Anthony\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=7>
  res = process_handler(cmd, _system_body)
C:\Users\Anthony\AppData\Roaming\Python\Python313\site-packages\IPython\utils\_process_win32.py:138: ResourceWarning: unclosed file <_io.BufferedReader name=8>
  res = process_handler(cmd, _system_body)


### Conectar a postgresql

In [79]:
import psycopg
import pandas as pd
import time

PG_HOST = "localhost"
PG_PORT = 5433
PG_DB = "vectordb"
PG_USER = "postgres"
PG_PASSWORD = "postgres"

conn = psycopg.connect(
    host=PG_HOST,
    port=PG_PORT,
    dbname=PG_DB,
    user=PG_USER,
    password=PG_PASSWORD
)

print("Conectado a PostgreSQL")

Conectado a PostgreSQL


### Habilitar pgvector y crear tabla

In [80]:
D = embeddings.shape[1]

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    
    cur.execute("DROP TABLE IF EXISTS documents;")
    
    cur.execute(f"""
        CREATE TABLE documents (
            id INTEGER PRIMARY KEY,
            text TEXT,
            embedding vector({D}),
            doc_id INTEGER,
            chunk_id INTEGER,
            source TEXT,
            category TEXT
        );
    """)
    
conn.commit()

print("Extensión pgvector habilitada")
print("Tabla documents creada con dimensión:", D)

Extensión pgvector habilitada
Tabla documents creada con dimensión: 384


### Preparar datos

In [81]:
def vector_to_sql(v):
    return "[" + ",".join(str(float(x)) for x in v) + "]"

pg_rows = []

for idx, row in chunks_df.iterrows():
    pg_rows.append((
        int(idx),
        str(row["text"]),
        vector_to_sql(embeddings[idx]),
        int(row["doc_id"]),
        int(row["chunk_id"]),
        str(row["source"]),
        "wikipedia"
    ))

print("Filas preparadas:", len(pg_rows))

Filas preparadas: 1976


### Insertar datos

In [82]:
insert_sql = """
    INSERT INTO documents (
        id, text, embedding, doc_id, chunk_id, source, category
    )
    VALUES (
        %s, %s, %s::vector, %s, %s, %s, %s
    );
"""

start = time.perf_counter()

with conn.cursor() as cur:
    cur.executemany(insert_sql, pg_rows)

conn.commit()

elapsed = time.perf_counter() - start

print("Filas insertadas:", len(pg_rows))
print("Tiempo de inserción:", elapsed, "segundos")

Filas insertadas: 1976
Tiempo de inserción: 0.4600563999993028 segundos


### Crear índice HNSW para cosine

In [83]:
with conn.cursor() as cur:
    cur.execute("""
        CREATE INDEX documents_embedding_hnsw_idx
        ON documents
        USING hnsw (embedding vector_cosine_ops);
    """)

conn.commit()

print("Índice HNSW creado para cosine distance")

Índice HNSW creado para cosine distance


### Consulta SQL Top-k

In [84]:
query_vector_sql = vector_to_sql(query_embedding[0])
k = 5

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            id,
            1 - (embedding <=> %s::vector) AS score,
            embedding <=> %s::vector AS distance,
            text,
            doc_id,
            chunk_id,
            source,
            category
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (query_vector_sql, query_vector_sql, query_vector_sql, k))
    
    rows = cur.fetchall()

for r in rows:
    print("=" * 100)
    print("ID:", r[0])
    print("Score:", r[1])
    print("Distance:", r[2])
    print("Metadata:", {
        "doc_id": r[4],
        "chunk_id": r[5],
        "source": r[6],
        "category": r[7]
    })
    print()
    print(r[3][:700])

ID: 1
Score: 0.8954437170526142
Distance: 0.10455628294738584
Metadata: {'doc_id': 1, 'chunk_id': 0, 'source': 'wikipedia', 'category': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 2
Score: 0.8741762788085667
Distance: 0.12582372119143326
Metadata: {'doc_id': 1, 'chunk_id': 1, 'source': 'wikipedia', 'category': 'wikipedia'}


### Funcion entregable

In [85]:
def pgvector_search(query_embedding, k=5):
    query_vector_sql = vector_to_sql(query_embedding[0])
    
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                id,
                1 - (embedding <=> %s::vector) AS score,
                embedding <=> %s::vector AS distance,
                text,
                doc_id,
                chunk_id,
                source,
                category
            FROM documents
            ORDER BY embedding <=> %s::vector
            LIMIT %s;
        """, (query_vector_sql, query_vector_sql, query_vector_sql, k))
        
        rows = cur.fetchall()
    
    results = []
    
    for row in rows:
        results.append({
            "id": row[0],
            "score": float(row[1]),
            "distance": float(row[2]),
            "text": row[3],
            "metadata": {
                "doc_id": row[4],
                "chunk_id": row[5],
                "source": row[6],
                "category": row[7]
            }
        })
    
    return results

### Probar funcion

In [86]:
pgvector_results = pgvector_search(query_embedding, k=5)

for r in pgvector_results:
    print("=" * 100)
    print("ID:", r["id"])
    print("Score:", r["score"])
    print("Distance:", r["distance"])
    print("Metadata:", r["metadata"])
    print()
    print(r["text"][:700])

ID: 1
Score: 0.8954437170526142
Distance: 0.10455628294738584
Metadata: {'doc_id': 1, 'chunk_id': 0, 'source': 'wikipedia', 'category': 'wikipedia'}

Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visual indication of the battery's state of charge. It is particularly important in the case of a battery electric vehicle. Some automobiles are fitted with a battery condition meter to monitor the starter battery. This meter is, essentially, a voltmeter but it may also be marked with coloured zones for easy visualization. Many newer cars no longer offer voltmeters or ammeters; instead, these vehicles typically have a light with the outline of an automotive battery on it. This can be somewhat misleading as it may be confused for an indicator of a b
ID: 2
Score: 0.8741762788085667
Distance: 0.12582372119143326
Metadata: {'doc_id': 1, 'chunk_id': 1, 'source': 'wikipedia', 'category': 'wikipedia'}


### Tabla

In [87]:
pgvector_df = pd.DataFrame([
    {
        "rank": i + 1,
        "id": r["id"],
        "score": r["score"],
        "distance": r["distance"],
        "doc_id": r["metadata"]["doc_id"],
        "chunk_id": r["metadata"]["chunk_id"],
        "source": r["metadata"]["source"],
        "category": r["metadata"]["category"],
        "text": r["text"][:300]
    }
    for i, r in enumerate(pgvector_results)
])

pgvector_df

,rank,id,score,distance,doc_id,chunk_id,source,category,text
0,1,1,0.895444,0.104556,1,0,wikipedia,wikipedia,Battery indicator A battery indicator (also kn...
1,2,2,0.874176,0.125824,1,1,wikipedia,wikipedia,ad battery when in reality it indicates a prob...
2,3,4,0.848882,0.151118,1,3,wikipedia,wikipedia,increase; in many cases the EMF remains more o...
3,4,1753,0.843664,0.156336,257,9,wikipedia,wikipedia,mping around large conductors and bus bars up ...
4,5,5,0.842319,0.157681,1,4,wikipedia,wikipedia,"otective diodes cannot be used, a battery will..."


## Parte 7: PostgreSQL + pgvector

En esta sección se utilizó PostgreSQL con la extensión `pgvector` para almacenar embeddings directamente en una tabla relacional. Se creó una tabla `documents` con columnas tradicionales como `id`, `text`, `doc_id`, `chunk_id`, `source` y `category`, además de una columna vectorial `embedding vector(D)`.

La búsqueda semántica se realizó mediante SQL, ordenando los documentos por distancia coseno usando el operador `<=>`. Conceptualmente, la consulta busca los documentos cuyo vector está más cerca del vector de la consulta.

También se creó un índice HNSW sobre la columna `embedding` usando `vector_cosine_ops`, con el objetivo de acelerar búsquedas aproximadas por similitud coseno.

### Preguntas PostgreSQL/pgvector

**¿Qué tan “explicable” te parece esta aproximación vs las otras?**  
Esta aproximación resulta bastante explicable porque todo se expresa con SQL. La búsqueda semántica se puede entender como una consulta que ordena filas por distancia entre vectores, usando un operador como `<=>` para distancia coseno.

**¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**  
PostgreSQL permite combinar búsqueda vectorial con operaciones relacionales tradicionales como `JOIN`, `WHERE`, `GROUP BY`, agregaciones, filtros por fechas, usuarios, categorías o permisos. Esto es útil cuando los embeddings forman parte de un sistema con datos estructurados.

**¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**  
Aunque pgvector es muy práctico, puede tener limitaciones frente a bases vectoriales dedicadas cuando el volumen de vectores, la concurrencia o los requisitos de baja latencia crecen mucho. Soluciones como Milvus, Weaviate o Qdrant están diseñadas específicamente para escalar búsquedas vectoriales y administrar índices especializados.

In [88]:
conn.close()